# Explain a padded-crop prediction

Run the cells on a curated photo to see the MEL-vs-NV prediction and a Grad-CAM overlay of the melanoma score. The overlay is aligned to the crop seen by the classifier.

The notebook loads the trained MEL-vs-NV heads from `models/mel-vs-nv-padded-crop.pth` in this checkout. Put the app's U-Net checkpoint at `models/unet_segmentation_full.pth`. To download the encoder on first use, accept the [DermFM-Zero-Open research terms](https://huggingface.co/redlessone/DermFM-Zero) and run `uv run --locked hf auth login` in the repository. The example photos are bundled in `demo/assets/demo-samples/`.

Encoder attribution · [Yan et al., DermFM-Zero](https://arxiv.org/abs/2602.10624)

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from torch.nn.functional import pad

cwd = Path.cwd().resolve()
repo_root = next(path for path in (cwd, *cwd.parents) if (path / "classifiers/export_padded_crop.py").is_file())
sys.path.insert(0, str(repo_root))

from classifiers.dermfm_zero import clinical_dtype, clinical_preprocess
from classifiers.encoders import load_rgb
from classifiers.export_padded_crop import PaddedCropTorchModel
from classifiers.lesion_crop import crop_with_skin_margin, frame_photo
from classifiers.segmentation import LesionSegmenter
from demo.samples import BUNDLED_SAMPLE_DIR

data_dir = Path(os.environ.get("MILK10K_DATA_DIR", repo_root / "data")).expanduser()
heads_path = repo_root / "models/mel-vs-nv-padded-crop.pth"
segmentation_path = repo_root / "models/unet_segmentation_full.pth"
photo_path = BUNDLED_SAMPLE_DIR / "skin-melanoma.jpg"

if not heads_path.is_file():
    raise FileNotFoundError(f"Missing the committed MEL-vs-NV heads at {heads_path}. Update your SkinVision checkout.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))

In [ ]:
model = PaddedCropTorchModel(heads_path, data_dir, device).eval()
segmenter = LesionSegmenter(segmentation_path, device)
photo = frame_photo(load_rgb(photo_path))
mask = segmenter.mask(photo)
crop = crop_with_skin_margin(photo, mask)
image = clinical_preprocess()(crop).unsqueeze(0).to(device=device, dtype=clinical_dtype(device))

with torch.no_grad():
    melanoma_score = model(image)[0, 1].item()

print("MEL" if melanoma_score >= model.cutoff else "NV")
print(f"Melanoma score {melanoma_score:.4f} · cutoff {model.cutoff:.4f}")
print("Padded crop" if mask.getbbox() is not None else "Original frame (empty mask)")

## Grad-CAM on the classifier input

The encoder divides the crop into patches. The reshape places patch activations back on the image grid after removing the extra classification token. The heatmap highlights regions associated with a higher melanoma score.

In [ ]:
grid_height, grid_width = model.encoder.patch_embed.dynamic_feat_size(tuple(image.shape[-2:]))
patch_height, patch_width = model.encoder.patch_embed.patch_size
# Match the encoder's padding so the heatmap can be trimmed to the source pixels.
padded_image = pad(image, (0, grid_width * patch_width - crop.width, 0, grid_height * patch_height - crop.height))


def reshape_transform(tokens):
    patches = tokens[:, 1:, :].reshape(tokens.shape[0], grid_height, grid_width, tokens.shape[-1])
    return patches.permute(0, 3, 1, 2)


with GradCAM(model=model, target_layers=[model.encoder.blocks[-1].norm1], reshape_transform=reshape_transform) as cam:
    heatmap = cam(input_tensor=padded_image.requires_grad_(True), targets=[ClassifierOutputTarget(1)])[0, : crop.height, : crop.width]

rgb = np.asarray(crop, dtype=np.float32) / 255.0
overlay = show_cam_on_image(rgb, heatmap, use_rgb=True)
figure, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, pixels, title in zip(axes, (crop, overlay), ("Classifier input", "Melanoma Grad-CAM"), strict=True):
    axis.imshow(pixels)
    axis.set_title(title)
    axis.axis("off")
plt.show()